In [ ]:
# =============================================================================
# jobs/31_build_critical_sites.py
# Critical sites for the exposure grid: schools, health, emergency services, prisons and utilities, in one standard table
# (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Interim sources, because AddressBase Premium (the national classification that would
# replace all of this) is not yet available to us. That access is the flagged major improvement.
#
# SOURCES (each one is optional and fails on its own: the notebook says what each gave)
#   gias   Department for Education, Get Information About Schools: open schools and colleges, with coordinates
#   ods    NHS Organisation Data Service API: NHS trust sites (hospitals and clinics) and GP practices, placed by postcode through the
#          ONS Postcode Directory in the lake
#   osm    OpenStreetMap UK file in the lake: hospitals and clinics not in the NHS list, care homes, colleges and nurseries, police,
#          fire, ambulance, coastguard and lifeboat stations, prisons, substations of 66 kV or more, power stations, sewage and water works
# Not covered by any of them: hazardous (COMAH) sites, and a complete care home list (the CQC feed was unavailable). Both are listed as gaps.
#
# OUTPUT  config.TBL_CRITICAL_SITES (restricted schema)
#   site_id, site_group (health, education, emergency, utilities), site_type, name, easting, northing, postcode, source, source_id,
#   lad (local authority district), tile, rofrs_rank (rivers and sea band 0 to 4), fz_rank, fz_sea_rank, fz_river_rank, fz_river_and_sea_rank
# The band at each site is read from the job 30 rasters, as for the address points. Job 34 counts this table per cell.
# Sites outside England are dropped (by the local authority boundaries). The same place from two sources is kept once: the NHS and DfE
# records win over OpenStreetMap within dedupe_m metres.
#
# MEMORY  Reading the UK OpenStreetMap file needs a few GB for the node locations: run it on a 64 GB node, not serverless.
#
# WHAT TO CHECK (printed): sites by source and group against rough national counts; sites in the rivers and sea High and Medium bands by group;
# example names from each group, so a wrong filter is obvious.
#
# Widgets: use_gias, use_ods, use_osm, gias_url, osm_file, include_osm_schools, dedupe_m, critical_source.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

# these are not on every cluster. They are installed if missing, and Python is restarted only if they still cannot be imported afterwards
# (a restart wipes everything defined so far, so the notebook would have to be run again from the top).
def _have():
    try:
        import osmium  # noqa: F401
        import pyproj  # noqa: F401
        import scipy  # noqa: F401
        import shapely  # noqa: F401
        return True
    except ImportError:
        return False


if not _have():
    get_ipython().run_line_magic("pip", "install -q osmium pyproj scipy shapely")
    import importlib
    importlib.invalidate_caches()
    if not _have():
        print("Packages installed but not importable in this session: restarting Python. Run all again from the top.")
        dbutils.library.restartPython()

import datetime as dt
import importlib
import io
import time

import numpy as np
import pandas as pd
import requests
import shapely
from pyspark.sql import functions as F

import config
import utils.address_points as _ap
import utils.critical_sites as _cs
import utils.osm_read as _osm
for _m in (_ap, _cs, _osm):
    importlib.reload(_m)
from utils.address_points import HAZARD_SOURCES, sample_hazard
from utils.critical_sites import COLUMNS, classify_osm, dedupe_nearby, gias_sites, ods_sites, osm_sites, standardise
from utils.helpers import get_spark
from utils.osm_read import read_osm_features

spark = get_spark()
HEADERS = {"User-Agent": "FGS-critical-sites"}
ONS = "/Volumes/prd_dash_bronze/ons_open_geography_portal_unrestricted"


def show(df):
    display(df.fillna("").astype(str) if len(df) else df)


dbutils.widgets.dropdown("critical_source", "interim", ["interim", "addressbase"], "Source (addressbase needs access we do not have yet)")
dbutils.widgets.dropdown("use_gias", "yes", ["yes", "no"], "Schools: DfE GIAS")
dbutils.widgets.dropdown("use_ods", "yes", ["yes", "no"], "Health: NHS ODS API")
dbutils.widgets.dropdown("use_osm", "yes", ["yes", "no"], "OpenStreetMap")
dbutils.widgets.text("gias_url", "", "GIAS file address (blank: try the dated file for recent days)")
dbutils.widgets.text("osm_file", f"/Volumes/prd_dash_bronze/openstreetmap_unrestricted/united_kingdom/format_PBF_united_kingdom/LATEST_united_kingdom/", "OpenStreetMap folder or .pbf file")
dbutils.widgets.dropdown("include_osm_schools", "no", ["no", "yes"], "Also take OpenStreetMap schools (GIAS is better)")
dbutils.widgets.text("dedupe_m", "150", "Drop an OpenStreetMap site within this many metres of an NHS or DfE site of the same group")
CRITICAL_SOURCE = dbutils.widgets.get("critical_source")
USE = {k: dbutils.widgets.get(f"use_{k}") == "yes" for k in ("gias", "ods", "osm")}
GIAS_URL, OSM_FILE = dbutils.widgets.get("gias_url").strip(), dbutils.widgets.get("osm_file").strip()
INCLUDE_OSM_SCHOOLS = dbutils.widgets.get("include_osm_schools") == "yes"
DEDUPE_M = float(dbutils.widgets.get("dedupe_m") or 150)
if CRITICAL_SOURCE == "addressbase":
    raise NotImplementedError("AddressBase Premium classification is not built: access to the restricted volume is still needed. Use interim.")

source_log = []
frames = {}

# Each source is saved to its own table as it is fetched. A source switched off is then read back from that copy, so one failed source can be
# fixed and re-run without fetching the others again (the OpenStreetMap read is the slow one). The final table is built from all of them.
from pyspark.sql.types import DoubleType, StringType, StructField, StructType
RAW_SCHEMA = StructType([StructField(c, DoubleType() if c in ("easting", "northing") else StringType()) for c in COLUMNS])


def stage_table(name):
    return f"{config.TBL_CRITICAL_SITES}_raw_{name}"


def save_stage(name, df):
    d = df.reindex(columns=COLUMNS).copy()
    for c in COLUMNS:
        if c not in ("easting", "northing"):
            d[c] = d[c].astype("object").where(d[c].notna(), None)
    spark.createDataFrame(d, schema=RAW_SCHEMA).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(stage_table(name))


def load_stage(name):
    """The saved copy of a source, or None if it has never been fetched."""
    if spark.catalog.tableExists(stage_table(name)):
        return spark.table(stage_table(name)).toPandas()
    return None


In [ ]:
# =============================================================================
# 1. SCHOOLS AND COLLEGES: DfE GIAS
# =============================================================================

if USE["gias"]:
    t0 = time.time()
    try:
        urls = [GIAS_URL] if GIAS_URL else [
            "https://ea-edubase-api-prod.azurewebsites.net/edubase/downloads/public/edubasealldata{d}.csv".format(d=(dt.date.today() - dt.timedelta(days=b)).strftime("%Y%m%d"))
            for b in range(0, 10)]
        raw = None
        for u in urls:
            r = requests.get(u, timeout=120, headers=HEADERS)
            if r.status_code == 200 and len(r.content) > 1_000_000:
                raw, used = r.content, u
                break
        if raw is None:
            raise RuntimeError("no GIAS file found at " + (GIAS_URL or "the dated addresses for the last 10 days") + ". Paste the address into gias_url.")
        df = pd.read_csv(io.BytesIO(raw), encoding="latin-1", low_memory=False)
        frames["gias"] = gias_sites(df)
        save_stage("gias", frames["gias"])
        source_log.append({"source": "gias", "status": "ok", "rows_in": len(df), "sites": len(frames["gias"]), "note": used, "seconds": round(time.time() - t0, 1)})
    except Exception as e:
        source_log.append({"source": "gias", "status": "FAILED", "note": str(e)[:200]})
print("1. GIAS:", [s for s in source_log if s["source"] == "gias"])
if not USE["gias"]:
    old = load_stage("gias")
    if old is not None:
        frames["gias"] = old
        source_log.append({"source": "gias", "status": "from saved copy", "sites": len(old)})



In [ ]:
# =============================================================================
# 2. HEALTH: NHS ODS API, placed by postcode
# Roles: RO198 NHS trust site (hospitals, clinics, health centres), RO76 GP practice. The API gives postcodes, not coordinates.
# =============================================================================

ROLES = (("RO198", "NHS trust site"), ("RO76", "GP practice"))


def fetch_role(role, page=1000):
    out, offset = [], 0
    while True:
        url = f"https://directory.spineservices.nhs.uk/ORD/2-0-0/organisations?PrimaryRoleId={role}&Status=Active&Limit={page}&Offset={offset}"
        r = requests.get(url, timeout=90, headers={**HEADERS, "Accept": "application/json"})      # without it the API answers 406 Not Acceptable
        r.raise_for_status()
        orgs = r.json().get("Organisations", [])
        out.extend(orgs)
        if len(orgs) < page:
            return out
        offset += page


if USE["ods"]:
    t0 = time.time()
    try:
        by_role = {label: fetch_role(role) for role, label in ROLES}
        all_codes = {o.get("PostCode", "").upper().replace(" ", "") for orgs in by_role.values() for o in orgs}
        pcs = (spark.read.parquet(f"{ONS}/ons_postcode_directory/format_PARQUET_ons_postcode_directory/LATEST_ons_postcode_directory/")
               .select("pcds", "east1m", "north1m")
               .filter(F.regexp_replace(F.upper("pcds"), " ", "").isin(list(all_codes))).toPandas())
        parts, notes = [], []
        for label, orgs in by_role.items():
            s = ods_sites(orgs, label, pcs)
            parts.append(s)
            notes.append(f"{label}: {len(orgs):,} organisations, {len(s):,} placed")
        frames["ods"] = standardise(parts)
        save_stage("ods", frames["ods"])
        source_log.append({"source": "ods", "status": "ok", "sites": len(frames["ods"]), "note": "; ".join(notes), "seconds": round(time.time() - t0, 1)})
    except Exception as e:
        source_log.append({"source": "ods", "status": "FAILED", "note": str(e)[:200]})
print("2. NHS ODS:", [s for s in source_log if s["source"] == "ods"])
if not USE["ods"]:
    old = load_stage("ods")
    if old is not None:
        frames["ods"] = old
        source_log.append({"source": "ods", "status": "from saved copy", "sites": len(old)})



In [ ]:
# =============================================================================
# 3. OPENSTREETMAP: emergency services, prisons, utilities, care homes, and health or education sites the lists above lack
# The UK file is large; reading it takes a while (progress is printed). Nodes are points; a way is the mean of its nodes.
# =============================================================================

if USE["osm"]:
    t0 = time.time()
    try:
        path = OSM_FILE
        if os.path.isdir(path):
            pbfs = [os.path.join(path, f) for f in os.listdir(path) if f.endswith(".pbf")]
            if not pbfs:
                raise RuntimeError(f"no .pbf file in {path}: {os.listdir(path)[:10]}")
            path = pbfs[0]
        print("Reading", path, "(this can take 15 to 40 minutes)")
        feats = read_osm_features(path, keep=lambda tags: classify_osm(tags, INCLUDE_OSM_SCHOOLS) is not None,
                                  progress=lambda seen, kept: print(f"   {seen:,} tagged objects read, {kept:,} critical sites kept, {time.time() - t0:.0f}s"))
        frames["osm"] = osm_sites(feats, INCLUDE_OSM_SCHOOLS)
        save_stage("osm", frames["osm"])
        source_log.append({"source": "osm", "status": "ok", "rows_in": len(feats), "sites": len(frames["osm"]), "note": path, "seconds": round(time.time() - t0, 1)})
    except Exception as e:
        source_log.append({"source": "osm", "status": "FAILED", "note": str(e)[:200]})
print("3. OpenStreetMap:", [s for s in source_log if s["source"] == "osm"])
if not USE["osm"]:
    old = load_stage("osm")
    if old is not None:
        frames["osm"] = old
        source_log.append({"source": "osm", "status": "from saved copy", "sites": len(old)})



In [ ]:
# =============================================================================
# 4. COMBINE, DROP DUPLICATES, KEEP ENGLAND, READ THE FLOOD BAND AT EACH SITE, WRITE
# =============================================================================

print("Sources:")
show(pd.DataFrame(source_log))
if not frames:
    raise RuntimeError("No source gave any sites. See the table above for why.")

official = standardise([frames.get("gias"), frames.get("ods")])
osm = frames.get("osm")
if osm is not None and len(osm):
    before = len(osm)
    osm = dedupe_nearby(official, osm, DEDUPE_M)
    print(f"OpenStreetMap sites: {before:,}; kept {len(osm):,} after dropping those within {DEDUPE_M:.0f} m of a DfE or NHS site of the same group.")
sites = standardise([official, osm])

# England only, and the local authority district, from the boundary file (sites outside it, such as Scotland and Wales, are dropped)
lad = spark.read.parquet(f"{ONS}/local_authority_districts_bdy_uk_bfe/format_GEOPARQUET_local_authority_districts_bdy_uk_bfe/LATEST_local_authority_districts_bdy_uk_bfe").select("LAD25CD", "geometry").toPandas()
lad = lad[lad["LAD25CD"].astype(str).str.startswith("E")].reset_index(drop=True)
geoms = shapely.from_wkb([bytes(g) for g in lad["geometry"]])
tree = shapely.STRtree(geoms)
pts = shapely.points(sites["easting"].to_numpy(), sites["northing"].to_numpy())
pi, gi = tree.query(pts, predicate="intersects")          # pairs: (site index, district index)
codes = lad["LAD25CD"].to_numpy(dtype=object)
lad_of = np.full(len(sites), None, dtype=object)
lad_of[pi] = codes[gi]                                      # a site on a shared edge takes one of the two
sites["lad"] = lad_of
n_all = len(sites)
sites = sites[sites["lad"].notna()].reset_index(drop=True)
print(f"England: {len(sites):,} of {n_all:,} sites ({n_all - len(sites):,} outside England dropped).")

sites = sample_hazard(sites, config.HAZARD_RASTER_DIR)
missing = [s for s in HAZARD_SOURCES if not os.path.isdir(os.path.join(config.HAZARD_RASTER_DIR, s))]
if missing:
    print("WARNING: no raster folder for", missing, "- those bands are 0. Run job 30 first.")

from pyspark.sql.types import ByteType, DoubleType, StringType, StructField, StructType
SCHEMA = StructType([StructField(c, StringType()) for c in ("site_id", "site_group", "site_type", "name")]
                    + [StructField("easting", DoubleType()), StructField("northing", DoubleType())]
                    + [StructField(c, StringType()) for c in ("postcode", "source", "source_id", "lad", "tile")]
                    + [StructField(c, ByteType()) for c in HAZARD_SOURCES.values()])
out = sites.reindex(columns=[f.name for f in SCHEMA])
for c in HAZARD_SOURCES.values():
    out[c] = out[c].astype("int8")
spark.createDataFrame(out, schema=SCHEMA).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(config.TBL_CRITICAL_SITES)
spark.sql(f"""COMMENT ON TABLE {config.TBL_CRITICAL_SITES} IS 'Critical sites in England from interim public sources: DfE Get Information About Schools, the NHS ODS API (placed by postcode) and OpenStreetMap, with the rivers and sea band and Flood Zone rank at each site from the job 30 rasters. Gaps: hazardous (COMAH) sites and a complete care home list. AddressBase Premium classification would replace this. Job 31.'""")
print(f"Written {len(out):,} sites to {config.TBL_CRITICAL_SITES}.")


In [ ]:
# =============================================================================
# 5. CHECKS
# Rough national counts for England, to catch a source that is missing or a filter that is wrong: schools about 24,000; GP practices about 6,500
# to 7,500; NHS trust sites a few thousand; fire stations about 1,000 to 1,500; police stations several hundred to 1,500; prisons about 120;
# ambulance stations several hundred; 66 kV substations a few thousand; sewage works several thousand; water treatment works about 1,000.
# =============================================================================

print("Sites by source and group:")
show(sites.groupby(["source", "site_group", "site_type"]).size().rename("sites").reset_index())
print("\nSites in the rivers and sea band (rank 3 Medium, 4 High) and in Flood Zone 3, by group:")
g = sites.assign(rs_high_medium=sites["rofrs_rank"] >= 3, in_fz3=sites["fz_rank"] == 2).groupby("site_group").agg(
    sites=("site_id", "size"), rs_high_medium=("rs_high_medium", "sum"), flood_zone_3=("in_fz3", "sum")).reset_index()
g["share_in_high_medium"] = (g["rs_high_medium"] / g["sites"]).round(3)
show(g)
print("\nExample names, five from each type:")
show(sites.groupby("site_type").head(5)[["site_group", "site_type", "name", "source", "lad"]].sort_values(["site_group", "site_type"]))


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success")
